# skyhunt — diagnostyka sesji zdjęć RAW (F0)

Sprawdza, co aparat zapisuje w plikach RAF: czas (z ułamkiem sekundy?), liczniki serii, czasy naświetlania,
kadencję bracketingu, bitowość, poziomy czerni i bieli, wzór X-Trans, czas dekodowania i przepustowość Drive.
Na tej podstawie zamrażamy model czasu dla analizy zdjęć. **Nic nie zapisuje** — tylko czyta pliki.

Sesja = podfolder w `MyDrive/skyhunt/raw/` (np. `dark_test_0210/`). Uruchom komórki od góry i wklej cały wydruk.

In [ ]:
# Drive i lista folderów z plikami RAF
import os, glob
from google.colab import drive
drive.mount('/content/drive')
RAW = '/content/drive/MyDrive/skyhunt/raw'
for d in sorted(os.listdir(RAW)):
    p = os.path.join(RAW, d)
    if os.path.isdir(p):
        n = len(glob.glob(f'{p}/*.RAF') + glob.glob(f'{p}/*.raf'))
        print(f'{d:30s} {n:5d} RAF')

In [ ]:
# Którą sesję sprawdzamy
SESSION = f'{RAW}/dark_test_0210'     # ← nazwa folderu z listy powyżej
N_DECODE = 3                          # ile plików zdekodować (rawpy)

In [ ]:
!pip -q install rawpy
!apt-get -qq install -y libimage-exiftool-perl > /dev/null
print('gotowe')

In [ ]:
# 1) EXIF całej sesji (exiftool): czas, ułamek sekundy, naświetlanie, liczniki
import subprocess, json, collections
import numpy as np
from datetime import datetime

files = sorted(glob.glob(f'{SESSION}/*.RAF') + glob.glob(f'{SESSION}/*.raf'))
sizes = [os.path.getsize(f) for f in files]
print(f'plików RAF: {len(files)}, razem {sum(sizes)/1e9:.2f} GB, średnio {np.mean(sizes)/1e6:.1f} MB/plik')

TAGS = ['FileName', 'DateTimeOriginal', 'SubSecTimeOriginal', 'SubSecDateTimeOriginal', 'ExposureTime',
        'ISO', 'FNumber', 'ShutterType', 'SequenceNumber', 'ImageCount', 'ImageNumber', 'ExposureCount',
        'AEBracketingSteps', 'BitsPerSample', 'RAWFileType', 'Compression', 'LongExposureNoiseReduction']
res = subprocess.run(['exiftool', '-j', '-n', '-a'] + [f'-{t}' for t in TAGS] + files,
                     capture_output=True, text=True)
ex = json.loads(res.stdout)
print()
print('— pierwsze 12 zdjęć —')
for e in ex[:12]:
    print({k: e.get(k) for k in TAGS if k in e})

print()
print('pola z Seq/Count/Bracket/Noise/Bit/SubSec/Shutter/Drive/Interval (1. plik):')
full = json.loads(subprocess.run(['exiftool', '-j', '-a', '-G1', files[0]],
                                 capture_output=True, text=True).stdout)[0]
KEYS = ('Seq', 'Count', 'Bracket', 'Noise', 'Bit', 'SubSec', 'Shutter', 'Drive', 'Continuous', 'Interval')
for k, v in full.items():
    if any(s in k for s in KEYS):
        print(f'  {k}: {v}')

In [ ]:
# 2) Kadencja: odstępy czasu EXIF (+ ułamek sekundy, jeśli jest) i czasy naświetlania
def t_of(e):
    t = datetime.strptime(str(e['DateTimeOriginal']), '%Y:%m:%d %H:%M:%S').timestamp()
    ss = e.get('SubSecTimeOriginal')
    return t + (float(f'0.{ss}') if ss not in (None, '') else 0.0)

ts = np.array([t_of(e) for e in ex])
exp = np.array([float(e.get('ExposureTime', 'nan')) for e in ex])
print(f'czas sesji: {ts[-1] - ts[0]:.1f} s')
print('odstępy między zdjęciami [s]:', collections.Counter(np.round(np.diff(ts), 2)).most_common(8))
print('czasy naświetlania [s]:', collections.Counter(np.round(exp, 4)).most_common(5))
print('sekwencja czasów (pierwsze 15):', list(np.round(exp[:15], 3)))
stamp = lambda e: str(e['DateTimeOriginal'])[11:] + ('.' + str(e['SubSecTimeOriginal'])
                                                      if e.get('SubSecTimeOriginal') not in (None, '') else '')
print('czasy EXIF (pierwsze 15):', [stamp(e) for e in ex[:15]])

In [ ]:
# 3) EXIF z JPEG-a osadzonego w RAF (Pillow) — czy da się czytać czas bez exiftool
import io, struct
from PIL import Image

with open(files[0], 'rb') as fh:
    head = fh.read(100)
    off, ln = struct.unpack('>II', head[84:92])
    fh.seek(off)
    jpg = fh.read(ln)
sub = Image.open(io.BytesIO(jpg)).getexif().get_ifd(0x8769)
print(f'RAF: magic={head[:16]!r}, JPEG offset={off}, len={ln}')
print('Pillow EXIF: DateTimeOriginal =', sub.get(0x9003), ' SubSec =', sub.get(0x9291),
      ' ExposureTime =', sub.get(0x829a))

In [ ]:
# 4) Dane z matrycy (rawpy): bitowość, czerń/biel, wzór X-Trans, czas dekodowania, szum
import time, rawpy

for f, e in list(zip(files, ex))[:N_DECODE]:
    t0 = time.time()
    with rawpy.imread(f) as r:
        img = r.raw_image_visible.copy()
        cols = r.raw_colors_visible[:6, :6].copy()
        bl, wl, desc = r.black_level_per_channel, r.white_level, r.color_desc
    dt = time.time() - t0
    c = img[1000:3000, 1500:4500].astype(np.float32)
    med = float(np.median(c))
    sig = 1.4826 * float(np.median(np.abs(c - med)))
    print()
    print(f'{os.path.basename(f)}  ExposureTime={e.get("ExposureTime")} s')
    print(f'  {img.shape} {img.dtype}, dekodowanie {dt:.2f} s, czerń {bl}, biel {wl}, opis kolorów {desc!r}')
    print(f'  max {img.max()}, mediana {med:.1f}, σ(MAD) {sig:.2f}, p99.99 {np.percentile(img, 99.99):.0f}')
    print('  wzór 6×6 (0=R, 1=G, 2=B):')
    print(cols)

In [ ]:
# 5) Przepustowość Drive: czytanie 5 plików
t0 = time.time()
n = 0
for f in files[5:10]:
    with open(f, 'rb') as fh:
        n += len(fh.read())
print(f'Drive: {n / 1e6 / (time.time() - t0):.0f} MB/s')